## Set Up

Connect to Neo4j

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase, RoutingControl

# Load variables from .env file
load_dotenv()

URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7687')
AUTH = (os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD'))

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connected to Neo4j successfully!")


Connected to Neo4j successfully!


Create function

In [2]:
# To create Node and Relationship
def add_friend(driver, name, friend_name): 
    driver.execute_query(
        'MERGE (a:Person {name: $name}) '
        'MERGE (friend:Person {name: $friend_name}) '
        'MERGE (a)-[:KNOWS]->(friend)',
        name=name, friend_name=friend_name, database_='neo4j',
    )

# To show 
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        'MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name '
        'RETURN friend.name ORDER BY friend.name',
        name=name, database_='neo4j', routing_=RoutingControl.READ,
    )
    for record in records:
        print(record['friend.name'])

# To run
add_friend(driver, 'Arthur', 'Guinevere')
add_friend(driver, 'Arthur', 'Lancelot')
add_friend(driver, 'Arthur', 'Merlin')
print_friends(driver, 'Arthur')


Guinevere
Lancelot
Merlin


### Checkpoint Questions 

1. **What is the name of the database used in this tutorial?**
   * **Answer:** `neo4j` (as specified by the argument `database_="neo4j"` in `driver.execute_query()`).

2. **What other access methods (such as Cypher DDL/DML syntax) can be used to access another database within the same instance?**
   * **Cypher `USE` clause:** Directs queries to a specific database, for example: `USE customers MATCH (n) RETURN n;`
   * **Cypher Administrative DDL:** Commands such as `SHOW DATABASES`, `CREATE DATABASE <name>`, `STOP DATABASE <name>`.
   * **Python Driver Parameter:** Specifying the target database in driver calls, such as `driver.session(database="system")` or `driver.execute_query(..., database_="db_name")`.

3. **What type of friendship graph was created here?**
   * **Directed Graph:** The relationship has an explicit direction from source to target: `(Arthur)-[:KNOWS]->(friend)`.
   * **Unweighted Graph:** The `KNOWS` relationship does not carry numeric weight properties.
   * **Star Graph / Ego Network:** "Arthur" acts as the central hub node with an out-degree of 3 connected to leaf nodes (Guinevere, Lancelot, Merlin), with no edges connecting the friends to each other.


### CYPHER Total Reset

In [3]:
FORCE_RESET = False

records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS total_nodes", database_="neo4j")
current_nodes = records[0]["total_nodes"]

if current_nodes > 10 and not FORCE_RESET:
    print(f"Database contains full dataset ({current_nodes:,} nodes). Reset skipped to protect data.")
else:
    print("Resetting database...")
    while True:
        records, _, _ = driver.execute_query(
            "MATCH (n) WITH n LIMIT 10000 DETACH DELETE n RETURN count(n) AS deleted;",
            database_="neo4j"
        )
        if records[0]["deleted"] == 0:
            break
    print("Database reset complete.")


Database contains full dataset (1,642,645 nodes). Reset skipped to protect data.


## Data Import from RetailRocket

In [4]:
import pandas as pd
import os

csv_file_path = 'data/events.csv'

# Check file existence
if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
else:
    print(f"File found: {csv_file_path}")

df_preview = pd.read_csv(csv_file_path, nrows=5)
df_preview


File found: data/events.csv


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [5]:
# 1. Create unique constraints so MERGE runs fast
def create_constraints(session):
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE;")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE;")

# 2. Cypher query
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)


In [6]:
from tqdm.auto import tqdm

# Check if database already has imported data
records, _, _ = driver.execute_query(
    "MATCH (v:Visitor) RETURN count(v) AS visitor_count", 
    database_="neo4j"
)
visitor_count = records[0]["visitor_count"]

if visitor_count > 1000:
    print(f"Neo4j already contains {visitor_count:,} visitors. Skipping import to prevent duplicates.")
else:
    batch_size = 5000
    total_rows = 2756101
    total_chunks = (total_rows // batch_size) + 1

    with driver.session(database="neo4j") as session:
        create_constraints(session)
        
        print(f"Read file from: {csv_file_path}")
        chunks = pd.read_csv(csv_file_path, chunksize=batch_size)

        print("Starting import to Neo4j...")
        for chunk in tqdm(chunks, total=total_chunks, desc="Import batch data"):
            chunk["transactionid"] = chunk["transactionid"].fillna("").astype(str)
            records = chunk.to_dict(orient="records")
            session.execute_write(import_batch, records)

    print("Import successful.")


Neo4j already contains 1,407,580 visitors. Skipping import to prevent duplicates.
